# Workshop 4 — Functions, Files & Exceptions
**Instructor:** Mario Lovrić · **Format:** live co-typing, type along in your own notebook.

### Today's plan
| # | Section | ~time |
|---|---|---|
| 1 | Functions — recap + lambda | 12 min |
| 2 | Working with files | 22 min |
| 3 | Exception handling | 15 min |
| — | Putting it together | 5 min |

**What we covered last time:** `if/elif/else`, logical operators, `for`/`while` loops, libraries (`math`, `statistics`), f-strings.

> Air-quality framing: we work with **PM2.5** CSV files and a real sensor dataset today.

## 1 · Functions — quick recap

From last session: `def`, arguments, `return`, defaults. One-minute warm-up.

In [ ]:
def is_bad(pm):
  return pm > 25

In [ ]:
is_bad(12)

In [ ]:
is_bad(42)

Default argument — used when the caller doesn't supply a value.

In [ ]:
def my_func(a=2,b=2):
  return a,b
my_func()

In [ ]:
def scale_pm(pm, factor=1.0):
    return pm * factor

In [ ]:
scale_pm(20)

In [ ]:
scale_pm(20, factor=2.5)

### Lambda — a one-liner function

Use when a function is short and only needed once.

```python
lambda arguments: expression
```

In [ ]:
double = lambda x: x * 2

In [ ]:
def double(x):
  return x*2

In [ ]:
double(10)

In [ ]:
square = lambda x: x ** 2

square(5)

In [ ]:
# try: square x in a lambda function and write in the traditional way (def)

In [ ]:
#List
a_list = [1, 2, 3, 4]
type(a_list)

In [ ]:
for item in a_list:
  print(item ** 2)

### List comprehensions

Apply an expression to every item in a list — in one line.

```python
[expression for item in list]
```

In [ ]:
readings = [10, 20, 30]
readings

In [ ]:
[x ** 2 for x in readings]

In [ ]:
[scale_pm(pm, 2.5) for pm in readings]

**Try it** — write a lambda `to_ugm3` that divides by 1000 (convert ng/m³ → µg/m³).

In [ ]:
# → your turn: lambda to_ugm3 that divides by 1000
# readings - apply - in list comprehension

In [ ]:
readings

In [ ]:
divide = lambda x: x/10

[divide(pm) for pm in readings]

## 2 · Working with files

### 2a · The write → read pattern

In Colab there is no pre-loaded filesystem — we create the file first.

`open(filename, mode)` modes:
| mode | meaning |
|---|---|
| `'w'` | write (creates / overwrites) |
| `'r'` | read (default) |
| `'a'` | append |

Always use `with` — it closes the file automatically even if something crashes.

In [ ]:
with open("pm25.csv", "w") as f:
    f.write("station,date,pm25\n")
    f.write("Zagreb,2023-01-01,18.5\n")
    f.write("Zagreb,2023-01-02,\n")
    f.write("Zagreb,2023-01-03,42.1\n")
    f.write("Rijeka,2023-01-01,12.3\n")
    f.write("Rijeka,2023-01-02,8.7\n")
    f.write("Rijeka,2023-01-03,\n")
print("written")

Read the whole file at once with `.read()`.

In [ ]:
with open("pm25.csv") as f:
    content = f.read()
print(content)

Read line by line with a `for` loop — works on files too large to fit in memory.

In [ ]:
with open("pm25.csv") as f:
    for line in f:
        print(line, end="")

### 2b · Real data from the internet

Download a real air-quality dataset with `urllib` (built-in, no install needed).

In [ ]:
import urllib.request

In [ ]:
url = "https://zenodo.org/records/6390135/files/Data.xlsx?download=1"

In [ ]:
urllib.request.urlretrieve(url, "Data.xlsx")
print("downloaded")

Open it with **pandas**.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel("Data.xlsx")

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.tail()

## 3 · Exception handling

Things go wrong: missing values, wrong filename, broken sensor.
`try/except` lets us handle errors without crashing.

```python
try:
    # risky code
except SomeError:
    # what to do when it fails
```

The problem — `float('')` raises `ValueError`:

In [ ]:
type('18.5')

In [ ]:
float('18.5')

In [ ]:
float('')   # → ValueError

Wrap it in `try/except`.

In [ ]:
value = ""
float(value)

In [ ]:
value = "86.7"

try:
    pm = float(value)
except ValueError:
    pm = -999

print(pm)

**Try it** — change `value` to `'42.1'` and re-run.

In [ ]:
# → your turn: value = '42.1', re-run the block above

### FileNotFoundError

In [ ]:
with open("missing.csv") as f:
        data = f.read()

In [ ]:
try:
    with open("missing.csv") as f:
        data = f.read()
except FileNotFoundError:
    print("NO FILE")

### Wrap the guard in a helper function

In [ ]:
def safe_float(measurement):
    """Convert string to float; return None if not possible."""
    try:
        return float(measurement)
    except ValueError:
        return None



In [ ]:
safe_float('18.5')

In [ ]:
safe_float('')

In [ ]:
readings = ['18.5', '', '42.1']

In [ ]:
for value in readings:
  safe_value = safe_float(value)
  print(safe_value)

## Putting it together

Read `pm25.csv`, parse each row safely, report per-station means.

In [ ]:
from statistics import mean

stations = {}

with open("pm25.csv") as f:
    next(f)
    for line in f:
        station, date, raw = line.split(",")
        pm = safe_float(raw)
        if pm is not None:
            if station not in stations:
                stations[station] = []
            stations[station].append(pm)

In [ ]:
for station, values in stations.items():
    print(f"{station}: mean = {mean(values):.1f} µg/m³  ({len(values)} valid readings)")

That's functions, files, and exceptions — the toolkit before we go deeper into **pandas** next session.